# P7 — K-Measurement: Alt 1 vs Alt 2

**Spørsmål:** Er K (Lovgiverens kapasitet) substrat-spesifikt eller universelt?

```
K  = Σ H(W_l)         spektral entropi av alle vektmatriser
C₀ = ρ × K            ρ = 0.8625437492
```

**Dom:**
- K_neo ≈ K_gpt2 → **Alt 2: K er universell**
- K_neo ≠ K_gpt2 → **Alt 1: K er substrat-spesifikt**

**Kjør alle celler fra topp til bunn.**

In [ ]:
# CELLE 1: Installer
!pip install -q transformers torch accelerate

In [ ]:
# CELLE 2: LIMFilter
import math, time, json
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class LIMFilter:
    def __init__(self):
        self.gamma  = 0.5772156649
        self.delta  = 4.6692016091
        self.zeta3  = 1.2020569032
        self.rho    = self.gamma / (self.delta - 4)   # 0.8625437492
        self.tau_lo = math.exp(-self.gamma)            # 0.5615
        self.tau_hi = 1.0 / self.zeta3                 # 0.8319

    def spectral_entropy(self, W):
        if W.numel() == 0 or W.ndim < 2:
            return 0.0
        try:
            Wf = W.float()
            md = min(Wf.shape)
            if md > 2048:
                _, S, _ = torch.svd_lowrank(Wf, q=min(512, md))
            else:
                _, S, _ = torch.linalg.svd(Wf, full_matrices=False)
            s = S.detach().cpu().numpy()
            s = s[s > 1e-10]
            if len(s) == 0:
                return 0.0
            p = s / s.sum()
            return float(-(p * np.log2(p + 1e-10)).sum())
        except Exception:
            return 0.0

    def compute_K(self, model):
        K, rows = 0.0, []
        for name, p in model.named_parameters():
            if p.ndim >= 2 and p.shape[0] > 1 and p.shape[1] > 1:
                H = self.spectral_entropy(p.detach())
                if H > 0:
                    K += H
                    rows.append((H, name, list(p.shape)))
        rows.sort(reverse=True)
        return K, rows

    def tau_dimless(self, hidden_states):
        bs, sl, hd = hidden_states.shape
        x = hidden_states.reshape(-1, hd).float()
        x = x - x.mean(0)
        L = (x.T @ x) / sl + 1e-6 * torch.eye(hd, device=x.device)
        ev = torch.linalg.eigvalsh(L)
        ev = ev[ev > 0]
        p  = ev / ev.sum()
        p  = p[p > 1e-10]
        return float(-(p * torch.log2(p)).sum())

lim = LIMFilter()
print(f"ρ  = {lim.rho:.10f}")
print(f"τ_min (dimless) = {lim.tau_lo:.4f}")
print(f"τ_max (dimless) = {lim.tau_hi:.4f}")
print("LIMFilter klar.")

In [ ]:
# CELLE 3: Mål K for en modell
def measure_model(model_name, num_tau_samples=10):
    print(f"\n{'='*60}")
    print(f"MODELL: {model_name}")
    print('='*60)

    print("Laster modell...")
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        output_hidden_states=True,
    )
    mdl.eval()
    cfg = mdl.config
    print(f"  hidden_dim={cfg.hidden_size}, layers={cfg.num_hidden_layers}, vocab={cfg.vocab_size}")
    print(f"  parametere: {sum(p.numel() for p in mdl.parameters())/1e6:.1f}M")

    # K fra vekter
    print("\nBeregner K...")
    t0 = time.time()
    K, rows = lim.compute_K(mdl)
    C0 = lim.rho * K
    print(f"  Ferdig på {time.time()-t0:.1f}s, {len(rows)} matriser")
    print(f"  K  = {K:.4f}")
    print(f"  C₀ = ρ×K = {C0:.4f} bits")
    print(f"  Topp 3:")
    for H, name, shape in rows[:3]:
        print(f"    {name}: H={H:.4f}, shape={shape}")

    # tau fra hidden states (dimensjonsløs)
    print(f"\nMåler τ ({num_tau_samples} samples)...")
    seed = "The coherence of a system is determined by its ability to maintain identity through constrained boundaries."
    device = next(mdl.parameters()).device
    taus = []
    for _ in range(num_tau_samples):
        inp = tok((seed * 4)[:256], return_tensors='pt', truncation=True, max_length=256)
        inp = {k: v.to(device) for k, v in inp.items()}
        with torch.no_grad():
            out = mdl(**inp, output_hidden_states=True)
        taus.append(lim.tau_dimless(out.hidden_states[-1]))
    tau = float(np.median(taus))
    in_zone = lim.tau_lo <= tau <= lim.tau_hi
    state = "COHERENCE" if in_zone else ("CHAOS" if tau < lim.tau_lo else "STASIS")
    print(f"  τ (dimensjonsløs) = {tau:.4f}")
    print(f"  Goldilocks-sone [{lim.tau_lo:.4f}, {lim.tau_hi:.4f}]: {state}")

    return {"model": model_name, "K": K, "C0": C0,
            "tau": tau, "state": state,
            "hidden_dim": cfg.hidden_size, "num_matrices": len(rows)}

print("Funksjon klar.")

In [ ]:
# CELLE 4: Kjør GPT-2 (baseline)
r_gpt2 = measure_model("gpt2")

In [ ]:
# CELLE 5: Kjør gpt-neo-1.3B (test — ingen gating, laster automatisk)
r_neo = measure_model("EleutherAI/gpt-neo-1.3B")

In [ ]:
# CELLE 6: Dom
print("\n" + "="*60)
print("SAMMENLIGNING OG DOM")
print("="*60)
print(f"  {'Modell':<30} {'K':>10} {'C₀':>10} {'τ':>8} {'Tilstand'}")
print(f"  {'-'*60}")
for r in [r_gpt2, r_neo]:
    print(f"  {r['model']:<30} {r['K']:>10.2f} {r['C0']:>10.2f} {r['tau']:>8.4f} {r['state']}")

k_ratio = r_neo['K'] / r_gpt2['K']
dim_ratio = r_neo['hidden_dim'] / r_gpt2['hidden_dim']

print(f"\n  K-forhold (neo/gpt2):     {k_ratio:.4f}x")
print(f"  Hidden dim-forhold:       {dim_ratio:.4f}x")
print(f"  GPT-2 baseline C₀ (P1):  4495.27")

print("\n" + "="*60)
print("DOM: Alt 1 vs Alt 2")
print("="*60)

if abs(k_ratio - 1.0) < 0.10:
    verdict = "ALT_2_UNIVERSAL"
    print(f"  >>> ALT 2: K er UNIVERSELL (ratio={k_ratio:.3f} ≈ 1.0)")
    print(f"  >>> Phi-loven er substrat-uavhengig!")
elif k_ratio > 1.10:
    verdict = "ALT_1_SCALES_UP"
    print(f"  >>> ALT 1: K skalerer OPP med arkitektur (ratio={k_ratio:.3f}x)")
    print(f"  >>> K er substrat-spesifikt — kalibreres per modell")
elif k_ratio < 0.90:
    verdict = "ALT_1_SCALES_DOWN"
    print(f"  >>> ALT 1: K skalerer NED med arkitektur (ratio={k_ratio:.3f}x)")
    print(f"  >>> K er substrat-spesifikt — kalibreres per modell")
else:
    verdict = "UNCLEAR"
    print(f"  >>> UKLAR: ratio={k_ratio:.3f}x — trenger flere modeller")

print("\nTofoo. Φ")

with open('p7_dom.json', 'w') as f:
    json.dump({'gpt2': r_gpt2, 'neo': r_neo, 'k_ratio': k_ratio,
               'dim_ratio': dim_ratio, 'verdict': verdict}, f, indent=2)
print("Lagret: p7_dom.json")